# Notebook 43 — Effective rank of the first-layer representation: one mediator for both architectures?

Notebook 37 showed that live filters mediate collapse in the CNN at fixed weight count. Notebook 42 showed that in the
MLP neither live units nor covered features alone do: damage is an inverted U in both, worst when every unit sees one
feature or every feature reaches three units, and every hand-built block is far worse than magnitude pruning at the same
weight count. The quantity that fits all of this is not a count but a dimension: how many independent directions of the
input the rest of the network can still see.

**Design.** For every saved model whose input layer was manipulated (CNN: conv.0 dose sweep, tap-distribution masks,
global and gradual recipes; MLP: input-layer dose sweep, the K x F grid, whole-network capacity sweep), a fixed sample
of 20,000 validation records is passed through the network and the representation entering the second prunable layer is
captured. Its effective dimensionality is summarised by the participation ratio of the covariance eigenvalues
(PR = (sum lambda)^2 / sum lambda^2) and by the number of principal components needed for 99% of the variance. The same
is computed for the penultimate representation. These are correlated with macro-F1 loss across all runs, pooled and
within each architecture, and compared with the two count-based predictors (surviving input weights, live channels).

**Gate (stated before running).** Effective rank is the shared mediator if, within each architecture, Spearman between
first-layer participation ratio and macro-F1 loss is at most -0.8, and pooled across both architectures it is at most -0.8
and more negative than the pooled correlation for surviving input weights. Any outcome is reported. CPU runtime.

In [ ]:
# --- Colab bootstrap (CPU is enough) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from src.config import CFG, PATHS
from src.data import load_raw, clean, temporal_within_capture_split
from src.train import load_anchor, feature_columns
from src import models as M
from src.comnet_audit import environment_record, write_json
torch.set_num_threads(max(1, os.cpu_count() or 1))
DATASET = 'ciciot2023'; SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed']); OUT = PATHS.tables('comnet')
KW = {'cnn1d': {'channels': (64, 128)}, 'mlp': {'hidden': (256, 128)}}; BASE = {'cnn1d': 'M0_paired', 'mlp': 'M0'}
N_SAMPLE = 20000
print('ready')

In [ ]:
# Data: a fixed stratified-by-position sample of VALIDATION records, scaled with the frozen training scaler
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR); feat_cols = feature_columns(df)
m0c, le, scaler, _ = load_anchor(DATASET, 'cnn1d', BASE['cnn1d'], ANCHOR, arch_kwargs=KW['cnn1d'])   # scaler is fit on train; identical for every model
rng = np.random.default_rng(0); vidx = np.array(splits['val']); pick = np.sort(rng.choice(vidx, size=min(N_SAMPLE, len(vidx)), replace=False))
Xv = torch.tensor(scaler.transform(df.loc[pick, feat_cols].to_numpy(np.float32)), dtype=torch.float32)
print(f'{len(df):,} rows | validation sample {Xv.shape[0]:,} x {Xv.shape[1]}')

In [ ]:
# Runs to analyse: (arch, cell, seed) with the macro-F1 loss from the committed tables
runs = []
isw = pd.read_csv(OUT / 'input_starvation_macro_f1_wide.csv'); m0 = isw[isw.cell == 'M0'].set_index(['arch', 'seed'])['test_macro_f1']
def add(arch, cell, seed, f1, family, note=''): runs.append({'arch': arch, 'cell': cell, 'seed': int(seed), 'family': family, 'note': note, 'test_macro_f1': float(f1), 'macro_f1_loss': float(m0[(arch, seed)] - f1)})
for _, r in isw[isw.cell != 'M0'].iterrows(): add(r.arch, r.cell, r.seed, r.test_macro_f1, 'input_dose', f'dose {r.dose}')
for _, r in pd.read_csv(OUT / 'tap_distribution_macro_f1_wide.csv').query("cell != 'M0'").iterrows(): add('cnn1d', f'{r.cell}_paired', r.seed, r.test_macro_f1, 'tap_mask', f'{r.mode} {r.taps}')
for _, r in pd.read_csv(OUT / 'gradual_macro_f1_wide.csv').query("cell != 'M0'").iterrows(): add('cnn1d', f'{r.cell}_paired', r.seed, r.test_macro_f1, 'gradual', r.cell)
cp = pd.read_csv(OUT / 'cnn_policy_macro_f1_wide.csv');
for _, r in cp[cp.cell == 'global80'].iterrows(): add('cnn1d', 'global80_paired', r.seed, r.test_macro_f1, 'policy', 'global80')
for _, r in pd.read_csv(OUT / 'mlp_grid_macro_f1_wide.csv').query("cell != 'M0'").iterrows(): add('mlp', r.cell, r.seed, r.test_macro_f1, 'grid', f'K{r.K} F{r.F} T{r["T"]}')
for _, r in pd.read_csv(OUT / 'mlp_capacity_sweep_macro_f1_wide.csv').query("cell != 'M0'").iterrows(): add('mlp', f'{r.cell}_mlp_paired', r.seed, r.test_macro_f1, 'capacity', r.cell)
runs = pd.DataFrame(runs); print(runs.groupby(['arch', 'family']).size().to_string()); print('total runs:', len(runs))

In [ ]:
# Effective-rank measures of the representation entering the second prunable layer, and of the penultimate features
def eff_rank(A):
    A = A - A.mean(0, keepdim=True); n = A.shape[0]
    cov = (A.T @ A) / (n - 1); lam = torch.linalg.eigvalsh(cov).clamp(min=0)
    pr = float(lam.sum() ** 2 / (lam ** 2).sum().clamp(min=1e-12)); cum = torch.cumsum(torch.flip(lam, [0]), 0) / lam.sum().clamp(min=1e-12)
    return pr, int((cum < 0.99).sum().item() + 1), int((lam > lam.max() * 1e-6).sum().item())

def layer_stats(arch, cell, seed):
    m = M.build(arch, len(feat_cols), len(le.classes_), **KW[arch]); ck = torch.load(PATHS.model(DATASET, arch, cell, seed), map_location='cpu', weights_only=False)
    m.load_state_dict(ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck); m.eval()
    second = m.conv[3] if arch == 'cnn1d' else m.body[3]                      # second prunable layer; its INPUT is the layer-1 representation
    first_w = (m.conv[0].weight if arch == 'cnn1d' else m.body[0].weight).detach()
    captured = {}
    h = second.register_forward_pre_hook(lambda mod, inp: captured.__setitem__('x', inp[0].detach()))
    with torch.no_grad():
        feats = torch.cat([m.features(Xv[i:i + 4096]) for i in range(0, len(Xv), 4096)])
        xs = []
        for i in range(0, len(Xv), 4096): m(Xv[i:i + 4096]); xs.append(captured['x'].flatten(1))
        rep1 = torch.cat(xs)
    h.remove()
    nz = (first_w != 0); live = int(nz.reshape(nz.shape[0], -1).any(dim=1).sum()); surviving = int(nz.sum())
    pr1, r99_1, rank1 = eff_rank(rep1.double()); prp, r99_p, rankp = eff_rank(feats.double())
    return {'surviving_input_weights': surviving, 'live_channels': live, 'rep1_dim': int(rep1.shape[1]), 'rep1_participation_ratio': pr1, 'rep1_rank99': r99_1, 'rep1_numerical_rank': rank1,
            'penult_participation_ratio': prp, 'penult_rank99': r99_p, 'penult_numerical_rank': rankp}

rows = []
for i, r in runs.iterrows():
    rows.append({**r.to_dict(), **layer_stats(r.arch, r.cell, r.seed)})
    if (i + 1) % 25 == 0: print(f'  {i + 1}/{len(runs)} models analysed')
per = pd.DataFrame(rows); per.to_csv(OUT / 'effective_rank_per_run.csv', index=False)
print(per.groupby(['arch', 'family'])[['surviving_input_weights', 'live_channels', 'rep1_participation_ratio', 'rep1_rank99', 'penult_participation_ratio', 'macro_f1_loss']].mean().round(3).to_string())

In [ ]:
# Correlations and gate
preds = ['surviving_input_weights', 'live_channels', 'rep1_participation_ratio', 'rep1_rank99', 'rep1_numerical_rank', 'penult_participation_ratio', 'penult_rank99']
def corr(d): return {p: float(spearmanr(d[p], d.macro_f1_loss).correlation) for p in preds}
res = {'pooled': corr(per), 'cnn1d': corr(per[per.arch == 'cnn1d']), 'mlp': corr(per[per.arch == 'mlp']),
       'mlp_grid_only': corr(per[per.family == 'grid']), 'cnn_tap_masks_only': corr(per[per.family == 'tap_mask'])}
summ = pd.DataFrame(res).T; summ.to_csv(OUT / 'effective_rank_correlations.csv'); print(summ.round(3).to_string())
pr = 'rep1_participation_ratio'
verdict = pd.DataFrame([
 {'criterion': 'cnn_rho_PR_vs_loss_le_-0.8', 'value': round(res['cnn1d'][pr], 3), 'pass': bool(res['cnn1d'][pr] <= -0.8)},
 {'criterion': 'mlp_rho_PR_vs_loss_le_-0.8', 'value': round(res['mlp'][pr], 3), 'pass': bool(res['mlp'][pr] <= -0.8)},
 {'criterion': 'pooled_rho_PR_vs_loss_le_-0.8', 'value': round(res['pooled'][pr], 3), 'pass': bool(res['pooled'][pr] <= -0.8)},
 {'criterion': 'pooled_PR_beats_surviving_weights', 'value': f"PR {res['pooled'][pr]:.3f} vs weights {res['pooled']['surviving_input_weights']:.3f}", 'pass': bool(res['pooled'][pr] < res['pooled']['surviving_input_weights'])},
 {'criterion': 'ref_mlp_grid_only_rho_PR', 'value': round(res['mlp_grid_only'][pr], 3), 'pass': ''},
 {'criterion': 'ref_penultimate_PR_pooled', 'value': round(res['pooled']['penult_participation_ratio'], 3), 'pass': ''},
])
print(); print(verdict.to_string(index=False))
print('\nEffective rank of the first-layer representation is the shared mediator:', bool(verdict[verdict['pass'] != '']['pass'].astype(bool).all()))
verdict.to_csv(OUT / 'effective_rank_gate_verdict.csv', index=False)
write_json(OUT / 'effective_rank_environment.json', {'n_sample': int(Xv.shape[0]), 'n_runs': int(len(per)), 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/43_effective_rank_mediator.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/effective_rank_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 43: effective rank of the first-layer representation across all input-manipulated runs (CNN + MLP); shared-mediator gate'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)